# IMDB Movie Recommendation Pipeline
Varje cell = ett lager. Byt modell i **Pipeline**-cellen, kör om.

In [ ]:
import json
from abc import ABC, abstractmethod
from pathlib import Path

import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

## 1. Data Loader
Läser CSV och normaliserar kolumnnamn oavsett Kaggle-variant.

In [ ]:
COLUMN_ALIASES = {
    "Series_Title": "title", "Title": "title", "name": "title",
    "Released_Year": "year",  "Year": "year",
    "IMDB_Rating":   "rating", "Rating": "rating", "imdb_rating": "rating",
    "Genre":         "genre",
    "Overview":      "overview", "Description": "overview", "description": "overview",
    "Director":      "director",
    "Runtime":       "runtime",
    "No_of_Votes":   "votes",   "Votes": "votes",
}

def load_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df.rename(columns={k: v for k, v in COLUMN_ALIASES.items() if k in df.columns})


# --- kör ---
df = load_df("movies.csv")
print(df.shape, df.columns.tolist())
df.head(3)

## 2. Watched-tracker
Persisterar i `watched.json`. Ändra listan manuellt här för att testa modellerna.

In [ ]:
WATCHED_FILE = Path("watched.json")

def load_watched() -> set:
    if WATCHED_FILE.exists():
        return set(json.loads(WATCHED_FILE.read_text()))
    return set()

def save_watched(watched: set) -> None:
    WATCHED_FILE.write_text(json.dumps(list(watched)))


watched = load_watched()
print(f"{len(watched)} sedda filmer")

## 3. Base Model
Alla modeller ärver `BaseRecommender`. Kräver `fit()` och `score()`.

In [ ]:
class BaseRecommender(ABC):
    @abstractmethod
    def fit(self, df: pd.DataFrame, watched: set) -> None: ...

    @abstractmethod
    def score(self, df: pd.DataFrame) -> pd.Series:
        """Returnera score per rad (högre = mer rekommenderad)."""
        ...

## 4. Cosine Similarity Model
TF-IDF på genre + overview → medelprofil av sedda filmer → cosine similarity mot alla.

In [ ]:
class CosineRecommender(BaseRecommender):
    def fit(self, df: pd.DataFrame, watched: set) -> None:
        text = (
            df.get("genre",    pd.Series("", index=df.index)).fillna("") + " " +
            df.get("overview", pd.Series("", index=df.index)).fillna("")
        ).str.strip()
        self._tfidf  = TfidfVectorizer()
        self._matrix = self._tfidf.fit_transform(text)
        mask = df["title"].isin(watched).values
        self._profile = self._matrix[mask].mean(axis=0)

    def score(self, df: pd.DataFrame) -> pd.Series:
        sims = cosine_similarity(self._profile, self._matrix).flatten()
        return pd.Series(sims, index=df.index)

## 5. Logistic Regression Model
Tränar en binär klassificerare: sedda filmer = 1, osedda = 0.
Features: TF-IDF text + normaliserade numeriska kolumner (rating, year, votes).

In [ ]:
class LogisticRecommender(BaseRecommender):
    def _text(self, df):
        return (
            df.get("genre",    pd.Series("", index=df.index)).fillna("") + " " +
            df.get("overview", pd.Series("", index=df.index)).fillna("")
        ).str.strip()

    def _features(self, df):
        text_f = self._tfidf.transform(self._text(df))
        if self._num_cols:
            num_f = self._scaler.transform(df[self._num_cols].fillna(0))
            return hstack([text_f, num_f])
        return text_f

    def fit(self, df: pd.DataFrame, watched: set) -> None:
        self._num_cols = [c for c in ["rating", "year", "votes"] if c in df.columns]
        self._tfidf = TfidfVectorizer(max_features=500)
        text_f = self._tfidf.fit_transform(self._text(df))

        if self._num_cols:
            self._scaler = StandardScaler(with_mean=False)
            num_f = self._scaler.fit_transform(df[self._num_cols].fillna(0))
            features = hstack([text_f, num_f])
        else:
            self._scaler = None
            features = text_f

        labels = df["title"].isin(watched).astype(int).values
        if labels.sum() == 0 or labels.sum() == len(labels):
            self._model = None
            return
        self._model = LogisticRegression(max_iter=1000, class_weight="balanced")
        self._model.fit(features, labels)

    def score(self, df: pd.DataFrame) -> pd.Series:
        if not getattr(self, "_model", None):
            return pd.Series(0.0, index=df.index)
        probs = self._model.predict_proba(self._features(df))[:, 1]
        return pd.Series(probs, index=df.index)

## 6. Ensemble
Normaliserar varje modells scores till [0,1] och tar viktat snitt.
Lägg till vilken `BaseRecommender` som helst med en vikt.

In [ ]:
class EnsembleRecommender(BaseRecommender):
    def __init__(self, models: list[tuple[BaseRecommender, float]]):
        self._models = models  # [(modell, vikt), ...]

    def fit(self, df: pd.DataFrame, watched: set) -> None:
        self._df, self._watched = df, watched
        for model, _ in self._models:
            model.fit(df, watched)

    def score(self, df: pd.DataFrame) -> pd.Series:
        total = pd.Series(0.0, index=df.index)
        for model, weight in self._models:
            s = model.score(df)
            rng = s.max() - s.min()
            total += weight * ((s - s.min()) / rng if rng > 0 else s)
        return total

    def recommend(self, n: int = 10) -> pd.DataFrame:
        unwatched = self._df[~self._df["title"].isin(self._watched)].copy()
        unwatched["score"] = self.score(self._df).loc[unwatched.index]
        return unwatched.nlargest(n, "score").drop(columns=["score"])

## 7. Pipeline
**Ändra här** för att byta vikter eller lägga till ny modell.

In [ ]:
pipeline = EnsembleRecommender([
    (CosineRecommender(),   0.6),
    (LogisticRecommender(), 0.4),
])

pipeline.fit(df, watched)
print("Pipeline tränad.")

## 8. Resultat

In [ ]:
display_cols = [c for c in ["title", "year", "rating", "genre", "director"] if c in df.columns]

recs = pipeline.recommend(n=10)
recs[display_cols]